# Frequency Allocation — Results Visualization

Loads CSVs produced by `FrequencyAllocationRuns.py` and reproduces the benchmark plots.

Run the script first:
```bash
python FrequencyAllocationRuns.py --test   # quick smoke test
python FrequencyAllocationRuns.py          # full run
```

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def load_csv(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    print(f"  missing: {path}")
    return pd.DataFrame()

df_rq        = load_csv('results_redqueen.csv')
df_ext_small = load_csv('results_ext_small.csv')
df_ext_large = load_csv('results_ext_large.csv')

for name, df in [("redqueen", df_rq), ("ext_small", df_ext_small), ("ext_large", df_ext_large)]:
    if not df.empty:
        print(f"{name}: {len(df)} rows, circuits: {sorted(df['circuit'].unique())}")

In [ ]:
config_order  = ['Standard SABRE', 'Standard MIRAGE', 'FASST', 'FINESSE']
device_order  = ['square_ring', 'square_ring_diag', 'square_ring_full']
device_labels = {'square_ring': 'Ring only', 'square_ring_diag': 'Ring + 1 diag', 'square_ring_full': 'Ring + full diag'}
colors        = ['#4C72B0', '#DD8452', '#1B6B3A', '#7B1D1D']
cfg_color     = dict(zip(config_order, colors))
markers       = {'square_ring': 'o', 'square_ring_diag': 's', 'square_ring_full': '^'}

metrics = [
    ('swaps',   'SWAPs',   'Mean SWAP count'),
    ('depth',   'Depth',   'Mean gate depth'),
    ('lf_cost', 'Σ –log F','Mean total log-fidelity cost'),
]

def plot_suite(df, title_prefix):
    circuits_list = sorted(df['circuit'].unique())
    summary = df.groupby(['circuit','config','device'])[['swaps','depth','lf_cost']].mean().reset_index()
    err     = df.groupby(['circuit','config','device'])[['swaps','depth','lf_cost']].std().reset_index()

    def get(tbl, dev, circ, cfg, col):
        row = tbl[(tbl.device==dev)&(tbl.circuit==circ)&(tbl.config==cfg)]
        return float(row[col].values[0]) if len(row) else 0.0

    n_cfg = len(config_order)
    w = 0.7 / n_cfg
    x = np.arange(len(circuits_list))

    # ── Bar charts for each metric ─────────────────────────────────────────────
    for col, ylabel, fig_title in metrics:
        if col not in df.columns:
            continue
        fig, axes = plt.subplots(1, len(device_order), figsize=(16, 4))
        fig.suptitle(f'{title_prefix} — {fig_title}', fontsize=12, fontweight='bold')
        for ax, dev in zip(axes, device_order):
            for k, (cfg, color) in enumerate(zip(config_order, colors)):
                vals = [get(summary, dev, c, cfg, col) for c in circuits_list]
                errs = [get(err,     dev, c, cfg, col) for c in circuits_list]
                ax.bar(x + (k-(n_cfg-1)/2)*w, vals, w, yerr=errs, color=color, label=cfg,
                       capsize=3, error_kw=dict(elinewidth=0.8, alpha=0.6))
            ax.set_title(device_labels[dev], fontsize=10)
            ax.set_xticks(x); ax.set_xticklabels(circuits_list, rotation=25, ha='right', fontsize=8)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            if ax is axes[0]: ax.set_ylabel(ylabel)
        axes[-1].legend(loc='upper right', fontsize=8, frameon=False)
        plt.tight_layout(); plt.show()

    # ── % improvement over SABRE ──────────────────────────────────────────────
    available_metrics = [(c, yl) for c, yl, _ in metrics if c in df.columns]
    fig, axes = plt.subplots(1, len(available_metrics), figsize=(5*len(available_metrics), 4))
    if len(available_metrics) == 1: axes = [axes]
    fig.suptitle(f'{title_prefix} vs Standard SABRE — % reduction', fontsize=11, fontweight='bold')
    for ax, (col, ylabel) in zip(axes, available_metrics):
        for i, (cfg, color) in enumerate(zip(config_order[1:], colors[1:])):
            pcts = []
            for dev in device_order:
                for c in circuits_list:
                    base = get(summary, dev, c, 'Standard SABRE', col)
                    val  = get(summary, dev, c, cfg, col)
                    if base > 0: pcts.append(100*(base-val)/base)
            ax.bar(i, np.mean(pcts), color=color, yerr=np.std(pcts), capsize=4,
                   error_kw=dict(elinewidth=0.9))
            ax.text(i, np.mean(pcts)+np.std(pcts)+0.3, f'{np.mean(pcts):.1f}%', ha='center', fontsize=8)
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax.set_xticks(range(len(config_order)-1))
        ax.set_xticklabels(config_order[1:], fontsize=8, rotation=15, ha='right')
        ax.set_ylabel(f'% reduction in {ylabel}')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout(); plt.show()

    # ── SWAP vs lf tradeoff ───────────────────────────────────────────────────
    summary2 = df.groupby(['device','circuit','config'])[['swaps','depth','lf_cost']].mean().reset_index()
    fig, ax = plt.subplots(figsize=(7, 6))
    for _, row in summary2[summary2.config == 'Standard SABRE'].iterrows():
        base_sw, base_lf = row.swaps, row.lf_cost
        dev, circ = row.device, row.circuit
        for cfg in config_order[1:]:
            r = summary2[(summary2.device==dev)&(summary2.circuit==circ)&(summary2.config==cfg)]
            if len(r) == 0: continue
            dx = 100*(r.swaps.values[0]-base_sw)/base_sw
            dy = 100*(r.lf_cost.values[0]-base_lf)/base_lf
            ax.scatter(dx, dy, color=cfg_color[cfg], marker=markers[dev], s=55, alpha=0.75, zorder=3)
    ax.axhline(0, color='black', linewidth=0.7, linestyle='--', zorder=1)
    ax.axvline(0, color='black', linewidth=0.7, linestyle='--', zorder=1)
    ax.text(0.02, 0.02, 'fewer SWAPs\nlower lf\n(win-win)', transform=ax.transAxes,
            ha='left', va='bottom', fontsize=7, color='green')
    cfg_handles  = [ax.scatter([],[],color=cfg_color[c],marker='o',s=55,label=c) for c in config_order[1:]]
    topo_handles = [ax.scatter([],[],color='grey',marker=markers[d],s=55,label=device_labels[d]) for d in device_order]
    leg1 = ax.legend(handles=cfg_handles, title='Config', fontsize=8, frameon=False, loc='upper left')
    ax.add_artist(leg1)
    ax.legend(handles=topo_handles, title='Topology', fontsize=8, frameon=False, loc='lower right')
    ax.set_xlabel('SWAP count change vs SABRE (%)')
    ax.set_ylabel('lf_cost change vs SABRE (%)')
    ax.set_title(f'{title_prefix} — routing tradeoff', fontsize=11)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout(); plt.show()

In [ ]:
if not df_rq.empty:
    plot_suite(df_rq, 'Red-queen circuits')
else:
    print("No red-queen data — run: python FrequencyAllocationRuns.py --suite redqueen")

In [ ]:
df_ext = pd.concat([df for df in [df_ext_small, df_ext_large] if not df.empty], ignore_index=True)
if not df_ext.empty:
    plot_suite(df_ext, 'Extended circuits')
else:
    print("No extended data — run: python FrequencyAllocationRuns.py --suite ext_small")

In [ ]:
df_all = pd.concat([df for df in [df_rq, df_ext_small, df_ext_large] if not df.empty], ignore_index=True)
if not df_all.empty:
    plot_suite(df_all, 'All circuits combined')
else:
    print("No data loaded.")